# 06 — Error Analysis (M5)

Model selection is now locked in: **tuned XGBoost** (`05_hyperparameter_tuning.ipynb`), val_r2=0.578,
no PCA (`04_feature_engineering.ipynb` — PCA tested and rejected, including a fair tuned-linear
check). This notebook studies *where* it's wrong, not just its overall score, per the M5 brief:
residuals, worst errors, near-identical configurations with different `y`, and performance by `X0`
level.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

from src.preprocessing import get_processed_data
from src.config import RANDOM_STATE

data = get_processed_data()

best_params = dict(max_depth=2, min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
                    learning_rate=0.05, n_estimators=200, reg_lambda=5, reg_alpha=0,
                    random_state=RANDOM_STATE)
model = XGBRegressor(**best_params)
model.fit(data.X_train, data.y_train)

preds = model.predict(data.X_val)
residuals = data.y_val.values - preds
print(f"val_r2 = {r2_score(data.y_val, preds):.4f}")
print(f"mean absolute error = {np.abs(residuals).mean():.2f} seconds")

## Residuals vs. predicted

![residuals](../reports/figures/06_residuals_vs_predicted.png)

Predictions cluster into distinct vertical bands rather than spreading smoothly across the range
-- a direct consequence of `max_depth=2`, the heavy regularization tuning required to fix
overfitting (very few possible outcomes per shallow tree). Residuals within each band skew
positive with a long tail (several 30-58 second underestimates), and almost no comparably large
overestimates -- **the model's main failure mode is underestimating unusually slow cars, not being
randomly wrong in both directions.**

In [ ]:
val_df = data.X_val.copy()
val_df["y_actual"] = data.y_val.values
val_df["y_pred"] = preds
val_df["abs_error"] = np.abs(residuals)

worst20 = val_df.sort_values("abs_error", ascending=False).head(20)
worst20[["y_actual", "y_pred", "abs_error"]]

## Worst 20 errors

All 20 are underestimates (`y_actual > y_pred`), several by 25-59 seconds, on cars whose real
bench time was unusually high (120-160+ seconds vs. the ~100s average). A decile check (predicted
value binned into 10 groups, mean residual per group) shows this is **not a broad systematic bias**
-- average residual per decile stays close to zero throughout the normal range. The issue is
isolated to a small number of severe misses on rare, unusually slow cars, not a general pattern
across all predictions.

In [ ]:
grouped = val_df.groupby("cat__X0").agg(n=("abs_error", "size"), mean_abs_error=("abs_error", "mean"))
grouped = grouped[grouped["n"] >= 3].sort_values("mean_abs_error", ascending=False)
print(f"Overall mean absolute error: {val_df['abs_error'].mean():.2f}")
grouped.head(10)

## Performance by `X0` category

![error by X0](../reports/figures/07_error_by_x0_group.png)

`X0` code 39 (letter `'s'`, 24 validation rows) has a mean absolute error of 11.95 seconds --
more than double the overall average of 5.33 -- a real, specific weak spot for the model, not just
noise (24 rows is a reasonable sample size, not a fluke of 1-2 points).

In [ ]:
X_full = pd.concat([data.X_train, data.X_val])
y_full = pd.concat([data.y_train, data.y_val])
full_df = X_full.copy()
full_df["y"] = y_full.values

feature_cols = X_full.columns.tolist()
full_df["_group_id"] = full_df.groupby(feature_cols).ngroup()

spread = full_df.groupby("_group_id")["y"].agg(["count", "min", "max"])
spread["range"] = spread["max"] - spread["min"]
spread = spread[spread["count"] > 1].sort_values("range", ascending=False)

print(f"Distinct 'identical configuration' groups (2+ rows, every feature matches): {len(spread)}")
print(f"Total rows involved: {spread['count'].sum()} out of {len(full_df)} ({spread['count'].sum()/len(full_df)*100:.1f}%)")
spread.head(10)

## Proof of an irreducible noise ceiling

**217 distinct groups, 515 rows (12.2% of the entire cleaned training set), share every single
feature value with at least one other row -- yet have different real bench times.** The most
extreme case: two cars, identical in every recorded feature, with bench times of 106.96s and
165.52s -- a 58.56 second difference **no model could ever resolve**, since there is nothing in
the data that distinguishes them. This directly confirms the EDA's early hypothesis and caps how
high any model built on these features alone can honestly score.

## Business insight (translating one finding for a non-technical audience)

**"Configuration alone (mainly `X0`) explains about 58% of why bench time varies between cars.
At least 12% of that variation is provably impossible to predict from the data we have -- cars
that look identical on paper genuinely take different amounts of time on the bench. That's not a
modeling weakness to fix; it's evidence that something outside this dataset -- likely which
production line, shift, operator, or day -- is driving part of the real-world variation. For
production planning: this model is a reliable average-case scheduling signal, but its confidence
should be treated with caution specifically for rare configuration codes (like `X0='s'`) and for
predicting unusually slow outliers -- and meaningfully improving beyond ~58% likely requires
capturing operational data this dataset doesn't include, not a better algorithm."**